# CDR-MLC Cluster-Conditioned Feature Augmentation

Diagnostic ablation that preserves raw classifier features and augments each hard-routed expert with:

- the causal 15-dimensional window vector;
- three centroid distances;
- three soft congestion memberships;
- the raw features multiplied by the membership of the selected expert.

The training partition and inference route remain identical to the leakage-safe hard CDR-MLC baseline. Test labels are used only for final metrics. This experiment does not replace the baseline.


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

MAIN = Path("CDR-MLC.ipynb")
if not MAIN.exists():
    MAIN = Path("CDR_MLC") / "CDR-MLC.ipynb"
namespace = {}
with MAIN.open(encoding="utf-8") as handle:
    notebook = json.load(handle)
exec(compile("".join(notebook["cells"][0]["source"]), str(MAIN), "exec"), namespace)
run_pipeline_from_two_files = namespace["run_pipeline_from_two_files"]
compute_sliding_window_stats = namespace["compute_sliding_window_stats"]

try:
    display
except NameError:
    display = print


def _membership(distances):
    inverse = 1.0 / np.maximum(distances, 1e-9) ** 2
    return inverse / inverse.sum(axis=1, keepdims=True)


def run_cluster_augmentation(train_file, test_file):
    baseline = run_pipeline_from_two_files(
        train_file, test_file, n_clusters=3, window_size=3,
        clustering_stats=["mean", "median", "std", "min", "max"],
    )
    train, test = baseline["train_df"], baseline["test_df"]
    features = baseline["classification_features"]
    target = baseline["target_column"]
    X_train = train[features].to_numpy(float)
    X_test = test[features].to_numpy(float)
    y_train = train[target].to_numpy()
    train_routes = train["cluster"].to_numpy()
    test_routes = test["cluster"].to_numpy()

    train_stats, _ = compute_sliding_window_stats(
        train[["SynAck", "AckDat", "TcpRtt"]],
        ["SynAck", "AckDat", "TcpRtt"], 3,
        ["mean", "median", "std", "min", "max"],
    )
    test_stats, _ = compute_sliding_window_stats(
        test[["SynAck", "AckDat", "TcpRtt"]],
        ["SynAck", "AckDat", "TcpRtt"], 3,
        ["mean", "median", "std", "min", "max"],
    )
    G_train = baseline["scaler"].transform(train_stats)
    G_test = baseline["scaler"].transform(test_stats)
    D_train = baseline["kmeans_model"].transform(G_train)
    D_test = baseline["kmeans_model"].transform(G_test)
    M_train = _membership(D_train)
    M_test = _membership(D_test)

    predictions = np.empty(len(X_test), dtype=int)
    experts = {}
    for cluster_id in range(3):
        train_mask = train_routes == cluster_id
        test_mask = test_routes == cluster_id
        augmented_train = np.hstack([
            X_train[train_mask],
            G_train[train_mask],
            D_train[train_mask],
            M_train[train_mask],
            X_train[train_mask] * M_train[train_mask, cluster_id, None],
        ])
        expert = RandomForestClassifier(
            n_estimators=80, random_state=42,
            class_weight="balanced", n_jobs=-1)
        expert.fit(augmented_train, y_train[train_mask])
        experts[cluster_id] = expert
        if test_mask.any():
            augmented_test = np.hstack([
                X_test[test_mask],
                G_test[test_mask],
                D_test[test_mask],
                M_test[test_mask],
                X_test[test_mask] * M_test[test_mask, cluster_id, None],
            ])
            predictions[test_mask] = expert.predict(augmented_test)

    y_test = test[target].to_numpy()
    augmented_metrics = {
        "accuracy": accuracy_score(y_test, predictions),
        "precision_weighted": precision_score(
            y_test, predictions, average="weighted", zero_division=0),
        "recall_weighted": recall_score(
            y_test, predictions, average="weighted"),
        "f1_weighted": f1_score(
            y_test, predictions, average="weighted"),
        "f1_macro": f1_score(y_test, predictions, average="macro"),
    }
    names = list(augmented_metrics)
    comparison = pd.DataFrame([
        {"method": "hard_cdr_mlc", **{
            name: baseline["test_results"][name] for name in names}},
        {"method": "cluster_augmentation", **augmented_metrics},
    ])
    for name in names:
        base = comparison.loc[
            comparison["method"] == "hard_cdr_mlc", name].iloc[0]
        comparison[f"{name}_gain_pp"] = 100 * (
            comparison[name] - base)
    display(comparison.round(4))
    return {
        "baseline": baseline, "experts": experts,
        "predictions": predictions, "comparison": comparison,
    }


In [ ]:
# scenario_1: run independently
scenario_1_augmentation = run_cluster_augmentation(
    "DATASETS/CDR-MLC/scale_1/Short/level_1.csv",
    "DATASETS/CDR-MLC/scale_1/Short/level_2.csv",
)


In [ ]:
# scenario_2: run independently
scenario_2_augmentation = run_cluster_augmentation(
    "DATASETS/CDR-MLC/scale_1/Short/level_1.csv",
    "DATASETS/CDR-MLC/scale_1/Short/level_3.csv",
)


In [ ]:
# scenario_3: run independently
scenario_3_augmentation = run_cluster_augmentation(
    "DATASETS/CDR-MLC/scale_1/Short/level_2.csv",
    "DATASETS/CDR-MLC/scale_1/Short/level_3.csv",
)


In [ ]:
# scenario_4: run independently
scenario_4_augmentation = run_cluster_augmentation(
    "DATASETS/CDR-MLC/scale_1/Short/CDR-MLC-Shuffle.csv",
    "DATASETS/CDR-MLC/scale_1/Long/CDR-MLC-Shuffle.csv",
)


In [ ]:
# scenario_5: run independently
scenario_5_augmentation = run_cluster_augmentation(
    "DATASETS/CDR-MLC/scale_1/Long/CDR-MLC-Shuffle.csv",
    "DATASETS/CDR-MLC/scale_1/Short/CDR-MLC-Shuffle.csv",
)
